## Declaração de bibliotecas e demais configurações

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import csv
import matplotlib.gridspec as gridspec

from sklearn import preprocessing

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier

#from matplotlib import rcParams
#rcParams['text.usetex'] = True
#rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'

np.random.seed(12345) #Deseja fixar a semente geradora de números aleatórios?

## Funções relacionadas ao método ou auxiliares

In [ ]:
#Função de leitura dos dados------------------------------------
def read_class_data(path):
    with open(path, newline='') as f:
        reader = csv.reader(f,delimiter=',')
        for row in reader:
            try:
                data = np.vstack( (data , np.asarray(row).astype(np.float) ) )
            except:
                data = np.asarray(row).astype(np.float)
    f.close()
    y = data[:,0]
    x = data[:,1:]
    return y,x

## Leitura dos dados (e normalização)
* Os dados considerados aqui foram obtidos do Observatório de Alagamento de Dartmounth (http://www.dartmouth.edu)

* Atributos são: Classe(indice-classe),Longitude,Latitute,Severidade, Duração (dias),Área afetada ($km^2$), Magnitude
* Maginitde = $log(Severidade \times Duracao \times Area\_afetada)$

* Classes: 
    * 1 - Chuva forte; (azul escuro)
    * 2 - Derretimento de gelo; (ciano)
    * 3 - Chuva torrencial; (verde)
    * 4 - Chuva de monção; (laranja)
    * 5 - Outros (ciclone, tornado, questões relacionadas a barragens e gelo) (magenta)

In [ ]:
#Dados de treinamento
path = 'FloodData.csv'

#Leitura dos dados
y,x_ = read_class_data(path)

#Normalização
x = np.copy(x_) #Cópia da variável
AtributosNormalizar = [2,3,4,5]
for i in AtributosNormalizar:
    mi = x[:,i].min()
    ma = x[:,i].max()
    a = 1/(ma-mi)
    b = -mi/(ma-mi)
    x[:,i] = a*x[:,i] + b

## Separação aleatória dos dados para treinamento e avaliação
* Veremos ainda processos mais sofisticados que este (apesar deste ser útil também!)
* É definida uma porcentagem destinada para avaliação, e restante é usado para treino
* Tal divisão é aleatoria:
    * Uma sequência de tamanho equivalente a quantidade de dados e gerada
    * Em seguida, tal sequência é ordenada e o índice que define a ordenação é considerado
    * A primeira parcela de P% dos dados é usado para avaliação e (100-P)% para treino

In [ ]:
#Gerar conjunto de treino e avaliação a partir de uma única fonte
N = y.shape[0]
percentAvalia = 0.33

#A ordenação dos valores aleatórios segundo seu índice/argumento
#O resultado é uma nova ordem aleatória (mais conveniente para este caso)
posAleatorias = np.argsort(np.random.uniform(0,1, y.shape[0] ))

#Subconjunto de avaliação
yI = y[ posAleatorias[0: np.int64(np.ceil(N*percentAvalia)) ] ]
xI = x[ posAleatorias[0: np.int64(np.ceil(N*percentAvalia)) ] , :]

#Subconjunto de treino
yD = y[ posAleatorias[np.int64(np.floor(N*percentAvalia)):-1] ]
xD = x[ posAleatorias[np.int64(np.floor(N*percentAvalia)):-1] , :]

## Visualizacão dos dados de treino/avaliação

In [ ]:
FS = (10,10) #Tamanho da figura a ser gerada
fig = plt.figure(constrained_layout=True,figsize=FS)
spec = gridspec.GridSpec(ncols=1, nrows=1, figure=fig)

ax = fig.add_subplot(spec[0,0])

ax.scatter( xD[:,0], xD[:,1], s=xD[:,5]*10, c='blue', alpha=0.5, label='Treino/Magnitude')
ax.scatter( xI[:,0], xI[:,1], s=xI[:,5]*10, c='red', alpha=0.5, label='Teste/Magnitude')
ax.set_xlabel('Longitude',fontsize=20)
ax.set_ylabel('Latitude',fontsize=20)

ax.set(xlim=(np.min(xD[:,0])-1,np.max(xD[:,0])+1), ylim=(np.min(xD[:,1])-1,np.max(xD[:,1])+1))
ax.set_aspect('equal', 'box')

ax.legend(fontsize=10)
ax.grid(True)

## Visualizacão dos dados de treino/classe

In [ ]:
#Usado na estilização do gráfico
indClasse = [1,2,3,4,5]
cores = ['blue', 'darkcyan', 'darkgreen', 'darkorange', 'magenta']
simbolos = ['s', 'd', 'D', '^', '*']
nomes = ['Chuva forte', 'Derretimento de gelo', 'Chuva torrencial', 'Chuva de monção', 'Outros']

FS = (10,10) #Tamanho da figura a ser gerada
fig = plt.figure(constrained_layout=True,figsize=FS)
spec = gridspec.GridSpec(ncols=1, nrows=1, figure=fig)

ax = fig.add_subplot(spec[0, 0])

for i,cor,simb,rotulo in zip(indClasse,cores,simbolos,nomes):
    pos = np.where(yD == i)[0]
    ax.scatter( xD[pos,0], xD[pos,1], s=xD[pos,5]*10, c=cor, marker=simb, alpha=0.5, label=rotulo)
    
ax.set_xlabel('Longitude',fontsize=20)
ax.set_ylabel('Latitude',fontsize=20)

ax.set(xlim=(np.min(xD[:,0])-1,np.max(xD[:,0])+1), ylim=(np.min(xD[:,1])-1,np.max(xD[:,1])+1))
ax.set_aspect('equal', 'box')

ax.legend()
ax.grid(True)

## Classificação dos dados
#### Diferentes quantidades de itens por folha (durante o treinamento)
* psi = {2, 10}
Outros parâmetros mantidos constantes - Observe!

In [ ]:
#Arquiteturas consideradas

#Instanciação dos classificadores
CART_2 = DecisionTreeClassifier(criterion='entropy', min_samples_split=2, min_impurity_decrease=10**(-7),random_state=1)
CART_10 = DecisionTreeClassifier(criterion='entropy', min_samples_split=10, min_impurity_decrease=10**(-7),random_state=1)
RF = RandomForestClassifier(n_estimators=100, criterion='entropy', min_samples_split=10, min_impurity_decrease=10**(-7), random_state=1)
AB_CART_50 = AdaBoostClassifier(DecisionTreeClassifier(min_samples_split=10,criterion='entropy',min_impurity_decrease=10**(-5),random_state=1), n_estimators=100, random_state=1)

#0- Longitude, 1 - Latitute (geralmente, não devemos usar...)
#2 - Severidade, 3 - Duração (dias), 4- Área afetada (𝑘𝑚2), 5 - Magnitude
atributos = [2,3,4,5] #Permite escolhar qual atributo usar
xD_sub = xD[:,atributos]
xI_sub = xI[:,atributos]

#Treinamento dos classificadores 
CART_2.fit(xD_sub,yD)
CART_10.fit(xD_sub,yD)
AB_CART_50.fit(xD_sub,yD)
RF.fit(xD_sub,yD)

## Avaliação do desempenho
* Análise baseada na concordância (%) entre Predição $vs$ Esperado.

In [ ]:
#Predição efetuada por cada uma das redes
yEst_cart2 = CART_2.predict(xI_sub)
yEst_cart10 = CART_2.predict(xI_sub)
yEst_abCart50 = AB_CART_50.predict(xI_sub)
yEst_rf = RF.predict(xI_sub)

#Concordâncias...
print('CART + Psi=2:               ', np.count_nonzero(yEst_cart2 == yI) / yI.shape[0])
print('CART + Psi=10:              ', np.count_nonzero(yEst_cart10 == yI) / yI.shape[0])
print('AdaBoos + CART (Estim=100): ', np.count_nonzero(yEst_abCart50 == yI) / yI.shape[0])
print('Random Forest (Estim=100):  ', np.count_nonzero(yEst_rf == yI) / yI.shape[0])

## Qual classe acerta/erra mais? (com relação à melhor classificação acima)
#### Escolha qual classificador deseja avaliar!

In [ ]:
nomes = ['Chuva forte','Derretimento de gelo','Chuva torrencial','Chuva de monção','Outros']

print('===Acurácias/classe -- CART + Psi=2===')
for i in range(0,5):
    posClasse = np.where(yI == (i+1))
    acc = np.count_nonzero(yEst_cart2[posClasse] == yI[posClasse]) / yI[posClasse].shape[0]
    print(nomes[i],': ',str(acc*100),'')

print('\n\n===Acurácias/classe -- Random Forest (Estim=100)===')
for i in range(0,5):
    posClasse = np.where(yI == (i+1))
    acc = np.count_nonzero(yEst_rf[posClasse] == yI[posClasse]) / yI[posClasse].shape[0]
    print(nomes[i],': ',str(acc*100),'')